In [ ]:
import torch
from torch.amp import autocast
import time
import matplotlib.pyplot as plt
from model import ContinuousMotionModel
from dataset.dataset import *
import utils.utils as utils
import soundfile as sf
from IPython.display import clear_output
from utils.performance_tracker import PerformanceTracker

device = utils.get_device()
sliding_tracker = PerformanceTracker()
standard_tracker = PerformanceTracker()

In [ ]:
# Get the newest model from the directory. That means find the newest folder, and then the newest file in that folder.
model_path = utils.get_latest_model_path("trained_models")
print(f"Model path: {model_path}")

# model_path = "models/sliding/sliding_seed_2_clean_20_denoise_50__ghz2rg4z_epoch_801.pth"
model_path = "models/normal/normal_seed_2__3aq6g23e_epoch_901.pth"

# Load the model
model: ContinuousMotionModel = ContinuousMotionModel.load_model(model_path, device)
model.condition_mask_probabilty = 0.0  # Disable condition mask probability for inference
model = model.to(device)
model.eval() # Set the model to evaluation mode

num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters in the model: {num_params}")

In [ ]:
import utils.animation.visualisation.new.animation_visualisation as animation_visualisation
print(animation_visualisation.init_visualization(display=False))

In [ ]:
# Sliding Diffusion inference loop
with autocast(device_type=device.type, dtype=torch.bfloat16):
    dataset = GPUDataset(
        consolidated_file="dataset/genea2023_dataset/val/main-agent/consolidated.npz",
        seq_length=70,
        seed_length=8,
        batch_size=1,
        epoch_length=1,  # Set to 1 for testing purposes
        return_audio_frame_index=True,  # Set to True to return the audio frame index
    )

    # Use no gradient calculation for inference
    with torch.no_grad():

        gesture_sequence, seed_gesture, _, main_agent_id_one_hot, finger_availability, start_frames = [
            item.to(device) for item in next(iter(dataset))
        ]
        full_audio_features = dataset.audio.to(device)
        start_frame = start_frames[0].item()  # Extract the first element from the tensor

        # Decode the input using the autoencoder model
        if model.pose_encoder is not None:
            encoded_gesture_sequence = model.pose_encoder.encode(gesture_sequence)
        else:
            encoded_gesture_sequence = gesture_sequence

        iteration_counter = 0
        
        while True:
            iteration_counter += 1
            # Start time for the current frame
            frame_start_time = time.time()

            # The audio features also have to be shifted by one frame
            # I have the full audio features and the starting frame, so I extract the audio features for the current frame
            actual_audio_features = full_audio_features[start_frame + iteration_counter: start_frame + iteration_counter + dataset.seq_length, :].unsqueeze(0)

            encoded_gesture_sequence, noisy_gesture_sequence = model.inference(encoded_gesture_sequence, actual_audio_features, main_agent_id_one_hot, finger_availability, seed_gesture)

            ########################################################################################################################################################################

            # Decode the output using the autoencoder model
            clear_output(wait=True)

            # In case the model added extra features for richer embeddings, we only take the original features for decoding
            denoised_frame_to_decode = encoded_gesture_sequence[:,model.diffusion.clean_frame_index,:model.original_pose_features_per_frame].unsqueeze(0)
            
            pre_decode_time = time.time()
            if model.pose_encoder is not None:
                unencoded_denoised_frame = model.pose_encoder.decode(denoised_frame_to_decode)
            else:
                unencoded_denoised_frame = denoised_frame_to_decode
            post_decode_time = time.time()
            
            print(f"Time taken for decoding in ms: {(post_decode_time - pre_decode_time) * 1000:.2f} ms")

            denmormalized_unencoded_denoised_frame = dataset.skeleton.denormalize_poses(unencoded_denoised_frame).squeeze(0).squeeze(0)
            # print(f"Shape of denormalized unencoded denoised frame: {denmormalized_unencoded_denoised_frame.shape}")
            animation_visualisation.send_pose(denmormalized_unencoded_denoised_frame.cpu(), dataset.skeleton)
            animation_visualisation.send_debug_tensor(torch.cat((actual_audio_features.squeeze(0).to(torch.float32),noisy_gesture_sequence.squeeze(0).to(torch.float32)), dim=1), "full tensor")


            print(f"Iteration: {iteration_counter}")
            frame_end_time = time.time()

            frame_time = frame_end_time - frame_start_time
            sliding_tracker.record_frame(frame_time)

            # Print the time taken for the current frame
            print(f"Frame {iteration_counter} processed in {frame_time:.4f} seconds ({1/(frame_end_time - frame_start_time):.2f} FPS)")

            # Sleep for the remaining time in the 30 FPS frame
            time_to_sleep = max(0, (1/30) - frame_time - 0.0005)  # 0.01 is a small buffer to account for processing time
            time.sleep(time_to_sleep)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

def visualize_performance(sliding_tracker, standard_tracker):
    fig, axes = plt.subplots(2, 1, figsize=(15, 10))
    
    # Get the first frame from the sliding tracker
    starting_timestamp = sliding_tracker.timestamps[0]

    # Subtract the starting frame time from all frame times to align them
    sliding_tracker.timestamps = [time - starting_timestamp for time in sliding_tracker.timestamps]

    starting_timestamp = standard_tracker.timestamps[0]
    standard_tracker.timestamps = [time - starting_timestamp for time in standard_tracker.timestamps]

    # 2. Timeline of frame times
    ax = axes[0]
    ax.plot(sliding_tracker.timestamps, sliding_tracker.frame_times, label="Sliding Diffusion", alpha=0.7)
    ax.plot(standard_tracker.timestamps, standard_tracker.frame_times, label="Standard Diffusion", alpha=0.7)
    ax.set_title("Frame Time Over Time")
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Frame Time (seconds)")
    ax.legend()
    
    # 3. Cumulative frames produced
    ax = axes[1]
    ax.plot(sliding_tracker.timestamps, sliding_tracker.cumulative_frames, label="Sliding Diffusion")
    ax.plot(standard_tracker.timestamps, standard_tracker.cumulative_frames, label="Standard Diffusion")
    ax.set_title("Cumulative Frames Over Time")
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Total Frames")
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("Sliding Diffusion Stats:")
    for k, v in sliding_tracker.get_stats().items():
        print(f"  {k}: {v}")
    
    print("\nStandard Diffusion Stats:")
    for k, v in standard_tracker.get_stats().items():
        print(f"  {k}: {v}")


visualize_performance(sliding_tracker, standard_tracker)

In [ ]:
# Normal Diffusion inference loop
with autocast(device_type=device.type, dtype=torch.bfloat16):
    dataset = GPUDataset(
        consolidated_file="dataset/genea2023_dataset/val/main-agent/consolidated.npz",
        seq_length=70,
        seed_length=8,
        batch_size=1,
        epoch_length=1,  # Set to 1 for testing purposes
        return_audio_frame_index=True,  # Set to True to return the audio frame index
    )

    # Use no gradient calculation for inference
    with torch.no_grad():

        gesture_sequence, seed_gesture, _, main_agent_id_one_hot, start_frames = [
            item.to(device) for item in next(iter(dataset))
        ]
        full_audio_features = dataset.audio.to(device)
        start_frame = start_frames[0].item()  # Extract the first element from the tensor

        # Decode the input using the autoencoder model
        # encoded_gesture_seed = model.pose_encoder.encode(gesture_sequence)

        iteration_counter = 0
        
        frame_start_time = time.time()
        while True:
            # Generate pure noise as the initial input
            denoised_gesture_sequence = torch.randn((1, model.gesture_length, model.pose_features_per_frame), device=device)
            
            actual_audio_features = full_audio_features[start_frame + iteration_counter * dataset.seq_length: start_frame + iteration_counter * dataset.seq_length + dataset.seq_length, :].unsqueeze(0)

            for timestep in range(model.diffusion.number_of_timesteps-1, -1, -1):

                # apply diffusion at the current timestep
                noisy_gesture_sequence = model.diffusion.forward(denoised_gesture_sequence, timestep)

                # Now we apply the model to denoise the gesture sequence
                timestep_tensor = torch.tensor([timestep], dtype=torch.int64, device=device)
                denoised_gesture_sequence = model.forward(
                    timestep=timestep_tensor,
                    one_hot_style=main_agent_id_one_hot,
                    audio_features=actual_audio_features,
                    noisy_gesture_sequence=noisy_gesture_sequence,
                    seed_gesture_sequence=seed_gesture
                )
                animation_visualisation.send_debug_tensor(torch.cat((actual_audio_features.squeeze(0).to(torch.float32),noisy_gesture_sequence.squeeze(0).to(torch.float32)), dim=1), "full tensor")

            ########################################################################################################################################################################

            # Decode the output using the autoencoder model
            clear_output(wait=True)

            # In case the model added extra features for richer embeddings, we only take the original features for decoding
            sequence_to_decode = denoised_gesture_sequence[..., :model.original_pose_features_per_frame]
        
            if model.pose_encoder is not None:
                unencoded_denoised_gesture_sequence = model.pose_encoder.decode(sequence_to_decode)
            else:
                unencoded_denoised_gesture_sequence = sequence_to_decode

            denmormalized_unencoded_denoised_gesture_sequence = dataset.skeleton.denormalize_poses(unencoded_denoised_gesture_sequence).squeeze(0).squeeze(0)

            animation_visualisation.send_debug_tensor(torch.cat((actual_audio_features.squeeze(0).to(torch.float32),denoised_gesture_sequence.squeeze(0).to(torch.float32)), dim=1), "full tensor")

            for frame in denmormalized_unencoded_denoised_gesture_sequence:
                # Start time for the current frame
                # Send each frame to the animation visualisation
                animation_visualisation.send_pose(frame.cpu(), dataset.skeleton)
                frame_end_time = time.time()

                frame_time = frame_end_time - frame_start_time
                standard_tracker.record_frame(frame_time)

                # Print the time taken for the current frame
                # print(f"Frame {iteration_counter} processed in {frame_end_time - frame_start_time:.4f} seconds ({1/(frame_end_time - frame_start_time):.2f} FPS)")

                # Sleep for the remaining time in the 30 FPS frame
                time_to_sleep = max(0, (1/30) - (frame_time) - 0.0005)  # 0.01 is a small buffer to account for processing time
                time.sleep(time_to_sleep)
                frame_start_time = time.time()

            iteration_counter += 1
            print(f"Iteration: {iteration_counter}")

In [ ]:
# Outpaint Diffusion inference loop
with autocast(device_type=device.type, dtype=torch.bfloat16):
    dataset = GPUDataset(
        consolidated_file="dataset/genea2023_dataset/val/main-agent/consolidated.npz",
        seq_length=70,
        seed_length=8,
        batch_size=1,
        epoch_length=1,  # Set to 1 for testing purposes
        return_audio_frame_index=True,  # Set to True to return the audio frame index
    )

    # Use no gradient calculation for inference
    with torch.no_grad():

        gesture_sequence, seed_gesture, _, main_agent_id_one_hot, start_frames = [
            item.to(device) for item in next(iter(dataset))
        ]
        full_audio_features = dataset.audio.to(device)
        start_frame = start_frames[0].item()  # Extract the first element from the tensor

        # Decode the input using the autoencoder model
        # encoded_gesture_seed = model.pose_encoder.encode(gesture_sequence)

        iteration_counter = 0
        
        current_sequence = gesture_sequence

        frame_start_time = time.time()
        while True:
            # Generate pure noise as the initial input
            
            overlap_frames = model.diffusion.overlap_frames

            current_sequence[:, :overlap_frames] = current_sequence[:, -overlap_frames:]
            current_sequence[:, overlap_frames:] = 0.0

            actual_audio_features = full_audio_features[start_frame + iteration_counter * dataset.seq_length: start_frame + iteration_counter * dataset.seq_length + dataset.seq_length, :].unsqueeze(0)

            for timestep in range(model.diffusion.number_of_timesteps-1, -1, -1):

                # apply diffusion at the current timestep
                noisy_gesture_sequence = model.diffusion.forward(current_sequence, timestep)

                # Now we apply the model to denoise the gesture sequence
                timestep_tensor = torch.tensor([timestep], dtype=torch.int64, device=device)
                denoised_gesture_sequence = model.forward(
                    timestep=timestep_tensor,
                    one_hot_style=main_agent_id_one_hot,
                    audio_features=actual_audio_features,
                    noisy_gesture_sequence=noisy_gesture_sequence,
                    seed_gesture_sequence=seed_gesture
                )
                animation_visualisation.send_debug_tensor(torch.cat((actual_audio_features.squeeze(0).to(torch.float32),noisy_gesture_sequence.squeeze(0).to(torch.float32)), dim=1), "full tensor")

            ########################################################################################################################################################################
            current_sequence = denoised_gesture_sequence
            # Decode the output using the autoencoder model
            clear_output(wait=True)

            # In case the model added extra features for richer embeddings, we only take the original features for decoding
            sequence_to_decode = denoised_gesture_sequence[:,model.diffusion.overlap_frames:, :model.original_pose_features_per_frame]
        
            if model.pose_encoder is not None:
                unencoded_denoised_gesture_sequence = model.pose_encoder.decode(sequence_to_decode)
            else:
                unencoded_denoised_gesture_sequence = sequence_to_decode

            denmormalized_unencoded_denoised_gesture_sequence = dataset.skeleton.denormalize_poses(unencoded_denoised_gesture_sequence).squeeze(0).squeeze(0)

            animation_visualisation.send_debug_tensor(torch.cat((actual_audio_features.squeeze(0).to(torch.float32),denoised_gesture_sequence.squeeze(0).to(torch.float32)), dim=1), "full tensor")

            for frame in denmormalized_unencoded_denoised_gesture_sequence:

            denmormalized_unencoded_denoised_gesture_sequence = dataset.skeleton.denormalize_poses(unencoded_denoised_gesture_sequence).squeeze(0).squeeze(0)

            animation_visualisation.send_debug_tensor(torch.cat((actual_audio_features.squeeze(0).to(torch.float32),denoised_gesture_sequence.squeeze(0).to(torch.float32)), dim=1), "full tensor")

            for frame in denmormalized_unencoded_denoised_gesture_sequence:
                # Start time for the current frame
                # Send each frame to the animation visualisation
                animation_visualisation.send_pose(frame.cpu(), dataset.skeleton)
                frame_end_time = time.time()

                frame_time = frame_end_time - frame_start_time
                standard_tracker.record_frame(frame_time)

                # Print the time taken for the current frame
                # print(f"Frame {iteration_counter} processed in {frame_end_time - frame_start_time:.4f} seconds ({1/(frame_end_time - frame_start_time):.2f} FPS)")

                # Sleep for the remaining time in the 30 FPS frame
                time_to_sleep = max(0, (1/30) - (frame_time) - 0.0005)  # 0.01 is a small buffer to account for processing time
                time.sleep(time_to_sleep)
                frame_start_time = time.time()

            iteration_counter += 1
            print(f"Iteration: {iteration_counter}")